# 05 · Binary Search

> Dimezzi lo spazio di ricerca a ogni passo: `O(log n)`. Funziona su array ordinati, ma anche su qualsiasi **predicato monotono** ("con capacità `x` ce la faccio?": falso, falso, …, vero, vero).
> Idee per i singoli problemi: [Pattern algoritmici › Ricerca binaria](../Fundamentals/Pattern%20algoritmici.md#ricerca-binaria)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Binary Search | template classico `l <= r` | `mid = (l + r) // 2` |
| Search a 2D Matrix | la matrice è una lista ordinata "srotolata" | `r, c = divmod(mid, cols)` |
| Koko Eating Bananas | binaria **sulla risposta**: minima velocità che basta | `math.ceil(a / b)` o `-(-a // b)`, `sum(...)` |
| Find Minimum in Rotated Sorted Array | confronta `nums[mid]` con `nums[r]` per capire dove sta il minimo | template `l < r` |
| Search in Rotated Sorted Array | una delle due metà è sempre ordinata: controlla se il target è lì | confronti con `nums[l]`, `nums[mid]`, `nums[r]` |
| Time Based Key-Value Store | timestamp crescenti per chiave → ultimo `≤ t` | `defaultdict(list)`, `bisect_right` |
| Median of Two Sorted Arrays | binaria sulla partizione dell'array più corto | `float("-inf")`, `float("inf")`, `max`/`min` |

## Sintassi al volo

| Serve | Sintassi |
|---|---|
| punto medio | `mid = (l + r) // 2` (in Python non c'è overflow) |
| prima posizione con `a[i] >= x` | `bisect.bisect_left(a, x)` |
| prima posizione con `a[i] > x` | `bisect.bisect_right(a, x)` |
| ultimo elemento `<= x` | `i = bisect_right(a, x) - 1` (se `i >= 0`) |
| bisect su un campo | `bisect_left(a, x, key=lambda item: item[0])` (3.10+) |
| divisione per eccesso | `math.ceil(a / b)` · `(a + b - 1) // b` · `-(-a // b)` |
| indice lineare ↔ cella | `r, c = divmod(i, cols)` · `i = r * cols + c` |

## Template 1: cerca un valore esatto (`l <= r`)

Lo spazio di ricerca è `[l, r]` inclusi; finisce quando è vuoto (`l > r`). A ogni passo escludi `mid`, quindi `l = mid + 1` o `r = mid - 1`.

In [1]:
def find_price(sorted_prices: list[int], target: int) -> int:
    l, r = 0, len(sorted_prices) - 1
    while l <= r:
        mid = (l + r) // 2
        if sorted_prices[mid] == target:
            return mid
        if sorted_prices[mid] < target:
            l = mid + 1
        else:
            r = mid - 1
    return -1  # qui l è la posizione di inserimento

print(find_price([300, 450, 990, 1200], 990), find_price([300, 450], 500))  # -> 2 -1

2 -1


## Template 2: primo indice che soddisfa un predicato (`l < r`)

Per "il minimo `x` tale che `ok(x)`": lo spazio `[l, r]` contiene sempre la risposta. Se `ok(mid)`, la risposta è `mid` o più a sinistra → `r = mid`; altrimenti `l = mid + 1`. Alla fine `l == r` è la risposta.

Con `r = mid` (non `mid - 1`) il ciclo deve essere `while l < r`, altrimenti non termina.

In [2]:
import math

# Capacità minima (pacchi/ora) per evadere tutti gli ordini entro 8 ore.
order_sizes = [3, 6, 7, 11]
hours = 8

def hours_needed(capacity: int) -> int:
    return sum(math.ceil(size / capacity) for size in order_sizes)

l, r = 1, max(order_sizes)  # limiti della risposta, non indici di un array
while l < r:
    mid = (l + r) // 2
    if hours_needed(mid) <= hours:
        r = mid      # mid basta: forse anche meno
    else:
        l = mid + 1  # mid non basta
print(l, hours_needed(l), hours_needed(l - 1))  # -> 4 8 10

4 8 10


**Divisione per eccesso:** `math.ceil(7 / 3)` usa i float (va bene fino a ~10¹⁵); `-(-7 // 3)` e `(7 + 3 - 1) // 3` restano interi.

In [3]:
import math
print(math.ceil(7 / 3), -(-7 // 3), (7 + 3 - 1) // 3)  # -> 3 3 3

3 3 3


## Matrice come array 1D

Se ogni riga è ordinata e il primo della riga successiva è maggiore dell'ultimo della precedente, la matrice è una lista ordinata di `rows * cols` elementi. `divmod(i, cols)` restituisce `(i // cols, i % cols)`.

In [4]:
matrix = [
    [1, 3, 5],
    [7, 9, 11],
]
rows, cols = len(matrix), len(matrix[0])
for i in (0, 4, 5):
    r, c = divmod(i, cols)
    print(i, (r, c), matrix[r][c])  # -> 0 (0, 0) 1 / 4 (1, 1) 9 / 5 (1, 2) 11

0 (0, 0) 1
4 (1, 1) 9
5 (1, 2) 11


## Array ruotati

Un array ordinato ruotato (`[4, 5, 6, 1, 2, 3]`) è fatto di due tratti crescenti. Due osservazioni bastano:
- se `nums[mid] > nums[r]`, il punto di rotazione (il minimo) è **a destra** di `mid`;
- tra `[l, mid]` e `[mid, r]`, almeno una metà è ordinata: la riconosci con `nums[l] <= nums[mid]`.

In [5]:
nums = [4, 5, 6, 1, 2, 3]
l, r = 0, len(nums) - 1
mid = (l + r) // 2
print(mid, nums[mid], nums[mid] > nums[r])  # -> 2 6 True: il minimo è a destra di mid
print(nums[l] <= nums[mid])                 # -> True: la metà sinistra [4, 5, 6] è ordinata
print(nums[l] <= 5 < nums[mid])             # -> True: confronti concatenati, il 5 sta a sinistra

2 6 True
True
True


## `bisect`: la ricerca binaria della libreria standard

Restituisce una **posizione di inserimento** che mantiene la lista ordinata. Per i problemi "valore più recente con timestamp `≤ t`" usa `bisect_right(...) - 1`.

In [6]:
from bisect import bisect_left, bisect_right, insort
from collections import defaultdict

price_history: defaultdict[str, list[tuple[int, int]]] = defaultdict(list)
price_history["MUG"] += [(1, 900), (5, 990), (9, 1050)]  # (timestamp, prezzo), timestamp crescenti

entries = price_history["MUG"]
i = bisect_right(entries, 6, key=lambda e: e[0]) - 1  # ultimo timestamp <= 6
print(i, entries[i][1] if i >= 0 else None)           # -> 1 990

print(bisect_left([10, 20, 20, 30], 20), bisect_right([10, 20, 20, 30], 20))  # -> 1 3
sorted_list = [10, 30]
insort(sorted_list, 20)
print(sorted_list)  # -> [10, 20, 30]  (inserimento O(n), ricerca O(log n))

1 990
1 3
[10, 20, 30]


⚠️ LeetCode usa una versione recente di Python, quindi `key=` funziona; se non fosse disponibile, cerca la tupla `(t, float("inf"))` per ottenere lo stesso risultato.

## Sentinelle infinite per le partizioni

Quando una partizione può essere vuota da un lato, usa `-inf`/`+inf` come valore "fuori dall'array": i confronti funzionano senza casi speciali.

In [7]:
a = [1, 3, 8]
cut = 0  # nessun elemento a sinistra del taglio
left_max = a[cut - 1] if cut > 0 else float("-inf")
right_min = a[cut] if cut < len(a) else float("inf")
print(left_max, right_min, left_max <= right_min)  # -> -inf 1 True
print((3 + 8) / 2, (3 + 8) // 2)  # -> 5.5 5: per la mediana pari usa /

-inf 1 True
5.5 5


## Errori comuni

- Mescolare i template: `while l <= r` con `r = mid` → ciclo infinito.
- Limiti sbagliati nella ricerca sulla risposta: `l` = minimo possibile (spesso 1, non 0 se dividi), `r` = massimo che sicuramente funziona.
- `bisect_left` vs `bisect_right` con duplicati: il primo va prima dei valori uguali, il secondo dopo.
- Dimenticare `- 1` dopo `bisect_right` quando cerchi "l'ultimo `≤ x`", e non controllare `i >= 0`.